# HOG + SVM Tuning (Colab)
Upload FER dataset as `data/fer2013/train/...` and `data/fer2013/test/...`, then run Cell 1 to Cell 3 in order.

In [ ]:
# Cell 1: Setup + imports + data loading
import os, json, pickle, itertools, subprocess, sys
from pathlib import Path
import numpy as np

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'scikit-image', 'scikit-learn', 'pandas', 'opencv-python'], check=False)

import cv2
import pandas as pd
from skimage.feature import hog
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
BASELINE_ACC = 0.4322931178601282
BASELINE_F1 = 0.38032291964402154

candidates = [Path('data/fer2013'), Path('data/fer203')]
DATA_ROOT = None
for c in candidates:
    if c.exists():
        DATA_ROOT = c
        break
if DATA_ROOT is None:
    raise FileNotFoundError('Dataset folder not found. Upload to data/fer2013 (preferred) or data/fer203.')

print(f'Using dataset root: {DATA_ROOT}')

def is_image_file(p: Path):
    return p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

def preprocess_image(img_u8, mode):
    if mode == 'none':
        return img_u8
    if mode == 'equalize':
        return cv2.equalizeHist(img_u8)
    if mode == 'clahe':
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(img_u8)
    raise ValueError(mode)

def load_folder_split(data_root: Path, val_fraction=0.1, seed=42):
    train_root = data_root / 'train'
    test_root = data_root / 'test'
    if not train_root.exists() or not test_root.exists():
        raise FileNotFoundError('Need data/fer2013/train and data/fer2013/test')

    x_train, y_train, x_val, y_val, x_test, y_test = [], [], [], [], [], []
    rng = np.random.default_rng(seed)

    for class_id, emo in enumerate(EMOTIONS):
        paths = [p for p in sorted((train_root / emo).iterdir()) if p.is_file() and is_image_file(p)]
        imgs = []
        for p in paths:
            img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                imgs.append(img)

        idx = np.arange(len(imgs))
        rng.shuffle(idx)
        n_val = int(round(len(imgs) * val_fraction))
        if n_val == 0 and len(imgs) > 1:
            n_val = 1
        if n_val >= len(imgs) and len(imgs) > 1:
            n_val = len(imgs) - 1

        val_idx = set(idx[:n_val])
        for i, im in enumerate(imgs):
            if i in val_idx:
                x_val.append(im); y_val.append(class_id)
            else:
                x_train.append(im); y_train.append(class_id)

    for class_id, emo in enumerate(EMOTIONS):
        paths = [p for p in sorted((test_root / emo).iterdir()) if p.is_file() and is_image_file(p)]
        for p in paths:
            img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                x_test.append(img); y_test.append(class_id)

    return (np.array(x_train, dtype=np.uint8), np.array(y_train, dtype=np.int64),
            np.array(x_val, dtype=np.uint8), np.array(y_val, dtype=np.int64),
            np.array(x_test, dtype=np.uint8), np.array(y_test, dtype=np.int64))

def extract_hog_batch(images, preprocess_mode, orientations, pixels_per_cell, cells_per_block):
    feats = []
    for im in images:
        im2 = preprocess_image(im, preprocess_mode)
        fd = hog(im2, orientations=orientations, pixels_per_cell=pixels_per_cell, cells_per_block=cells_per_block, block_norm='L2-Hys', feature_vector=True, visualize=False)
        feats.append(fd.astype(np.float32))
    return np.array(feats, dtype=np.float32)

X_train_img, y_train, X_val_img, y_val, X_test_img, y_test = load_folder_split(DATA_ROOT)
print('Loaded split sizes:', len(y_train), len(y_val), len(y_test))

In [ ]:
# Cell 2: Full HOG+SVM tuning run (180 configs)
preprocess_modes = ['none', 'equalize', 'clahe']
orientations_grid = [8, 9, 12]
ppc_grid = [(6, 6), (8, 8)]
cpb_grid = [(2, 2), (3, 3)]
linear_c = [0.5, 1.0, 2.0]
rbf_c = [1.0, 5.0]

configs = []
for mode, ori, ppc, cpb in itertools.product(preprocess_modes, orientations_grid, ppc_grid, cpb_grid):
    for c in linear_c:
        configs.append((mode, ori, ppc, cpb, 'linear', c))
    for c in rbf_c:
        configs.append((mode, ori, ppc, cpb, 'rbf', c))

print('Total configs:', len(configs))

feature_cache = {}
for mode, ori, ppc, cpb in itertools.product(preprocess_modes, orientations_grid, ppc_grid, cpb_grid):
    k = (mode, ori, ppc, cpb)
    feature_cache[(k, 'train')] = extract_hog_batch(X_train_img, mode, ori, ppc, cpb)
    feature_cache[(k, 'val')] = extract_hog_batch(X_val_img, mode, ori, ppc, cpb)
    feature_cache[(k, 'test')] = extract_hog_batch(X_test_img, mode, ori, ppc, cpb)

rows = []
best = None
best_model = None
best_scaler = None

for i, (mode, ori, ppc, cpb, clf_type, cval) in enumerate(configs, 1):
    k = (mode, ori, ppc, cpb)
    Xtr = feature_cache[(k, 'train')]
    Xva = feature_cache[(k, 'val')]
    Xte = feature_cache[(k, 'test')]

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xva_s = scaler.transform(Xva)
    Xte_s = scaler.transform(Xte)

    if clf_type == 'linear':
        clf = LinearSVC(C=cval, dual=False, max_iter=6000, class_weight='balanced', random_state=42)
    else:
        clf = SVC(C=cval, kernel='rbf', gamma='scale', class_weight='balanced')

    clf.fit(Xtr_s, y_train)
    yva = clf.predict(Xva_s)
    yte = clf.predict(Xte_s)

    va_acc = accuracy_score(y_val, yva)
    va_f1 = f1_score(y_val, yva, average='macro')
    te_acc = accuracy_score(y_test, yte)
    te_f1 = f1_score(y_test, yte, average='macro')
    cm = confusion_matrix(y_test, yte).tolist()

    rows.append({'preprocess': mode, 'orientations': ori, 'pixels_per_cell': str(ppc), 'cells_per_block': str(cpb), 'classifier': clf_type, 'c_value': cval, 'val_accuracy': va_acc, 'val_macro_f1': va_f1, 'test_accuracy': te_acc, 'test_macro_f1': te_f1})

    score = (va_f1, va_acc)
    if best is None or score > (best['val_macro_f1'], best['val_accuracy']):
        best = {'preprocess': mode, 'orientations': ori, 'pixels_per_cell': list(ppc), 'cells_per_block': list(cpb), 'classifier': clf_type, 'c_value': cval, 'val_accuracy': va_acc, 'val_macro_f1': va_f1, 'test_accuracy': te_acc, 'test_macro_f1': te_f1, 'confusion_matrix': cm}
        best_model = clf
        best_scaler = scaler

    if i % 10 == 0 or i == 1:
        print(f'[{i}/{len(configs)}] current best val_f1={best["val_macro_f1"]:.4f}, val_acc={best["val_accuracy"]:.4f}')

Path('reports').mkdir(parents=True, exist_ok=True)
Path('saved_models').mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(rows).sort_values(['val_macro_f1', 'val_accuracy'], ascending=False)
df.to_csv('reports/hog_svm_tuning_results.csv', index=False)
with open('reports/hog_svm_best_config.json', 'w') as f:
    json.dump({'best_config': best}, f, indent=2)

artifact = {'classifier': best_model, 'scaler': best_scaler, 'hog_config': {'orientations': best['orientations'], 'pixels_per_cell': best['pixels_per_cell'], 'cells_per_block': best['cells_per_block']}, 'preprocess': best['preprocess'], 'class_order': EMOTIONS, 'metrics': {'accuracy': best['test_accuracy'], 'macro_f1': best['test_macro_f1'], 'val_accuracy': best['val_accuracy'], 'val_macro_f1': best['val_macro_f1'], 'confusion_matrix': best['confusion_matrix']}}
with open('saved_models/hog_svm_artifact_tuned.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print('Saved: reports/hog_svm_tuning_results.csv, reports/hog_svm_best_config.json, saved_models/hog_svm_artifact_tuned.pkl')

In [ ]:
# Cell 3: Summary vs baseline
import json
import pandas as pd

with open('reports/hog_svm_best_config.json', 'r') as f:
    best = json.load(f)['best_config']
df = pd.read_csv('reports/hog_svm_tuning_results.csv')

tuned_acc = float(best['test_accuracy'])
tuned_f1 = float(best['test_macro_f1'])

print('=== HOG+SVM TUNED SUMMARY ===')
print('Best config:')
for k in ['preprocess', 'orientations', 'pixels_per_cell', 'cells_per_block', 'classifier', 'c_value']:
    print(f'  {k}: {best[k]}')

print('\nMetrics:')
print(f'  Baseline accuracy: {BASELINE_ACC:.4f} ({BASELINE_ACC*100:.2f}%)')
print(f'  Tuned accuracy:    {tuned_acc:.4f} ({tuned_acc*100:.2f}%)')
print(f'  Delta accuracy:    {tuned_acc-BASELINE_ACC:+.4f} ({(tuned_acc-BASELINE_ACC)*100:+.2f}%)')
print(f'  Baseline macro F1: {BASELINE_F1:.4f}')
print(f'  Tuned macro F1:    {tuned_f1:.4f}')
print(f'  Delta macro F1:    {tuned_f1-BASELINE_F1:+.4f}')

print('\nTop 10 configs by val_macro_f1:')
display(df.sort_values(['val_macro_f1', 'val_accuracy'], ascending=False).head(10))